# 03 - Fine-tune YOLOv8n on the firearm dataset (Colab, GPU)

## How to reproduce

1. Locally: `uv run python -m weapon_detection.ml.yolo_data` creates `outputs/yolo_colab/yolo_firearm.zip`.
2. Upload that zip to the top level of Google Drive (`MyDrive`).
3. In Colab: Runtime -> Change runtime type -> **T4 GPU**, then Run all.
4. Download `MyDrive/yolo_results/best.pt` to `models/firearm_yolov8n_best.pt`.

Seed 42 is passed to the trainer (`seed=42, deterministic=True`). Runtime depends on the GPU; Ultralytics prints the time per epoch.

**1. Mount Google Drive** so the notebook can read the zip and save results.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

**2. Unzip the dataset** to the fast local disk of the Colab machine.

In [ ]:
!unzip -q -o /content/drive/MyDrive/yolo_firearm.zip -d /content
!ls /content/yolo_firearm

**3. Install Ultralytics** (the YOLO library).

In [ ]:
!pip install -q ultralytics

**4. Fix the dataset path in data.yaml.** It still points to the folder on my laptop; YOLO needs the Colab folder.

In [ ]:
import yaml

path = '/content/yolo_firearm/data.yaml'
data = yaml.safe_load(open(path))
data['path'] = '/content/yolo_firearm'
yaml.safe_dump(data, open(path, 'w'), sort_keys=False)
print(open(path).read())

**5. Check the GPU.** Training on CPU would take many hours.

In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')

**6. Train.** Start from COCO-pretrained `yolov8n.pt` (transfer learning) with the same settings as `weapon_detection/ml/yolo_train.py`.

In [ ]:
from ultralytics import YOLO

args = dict(data='/content/yolo_firearm/data.yaml', epochs=30, imgsz=640, batch=16,
            patience=10, seed=42, deterministic=True, workers=2, plots=True,
            device=0, project='/content/yolo_runs', name='firearm_yolov8n')
print(args)
model = YOLO('yolov8n.pt')
model.train(**args)

**7. Copy the results back to Drive** (Colab's disk is deleted when the session ends).

In [ ]:
import os
import shutil

run = '/content/yolo_runs/firearm_yolov8n'
out = '/content/drive/MyDrive/yolo_results'
os.makedirs(out, exist_ok=True)
for name in ['weights/best.pt', 'results.csv', 'results.png', 'confusion_matrix.png',
             'args.yaml']:
    shutil.copy(os.path.join(run, name), out)
    print('copied', name)

**8. Show the training curves and the validation confusion matrix.**

In [ ]:
import pandas as pd
from IPython.display import Image, display

display(Image(run + '/results.png'))
display(Image(run + '/confusion_matrix.png'))
pd.read_csv(run + '/results.csv').tail()